<left>
    <img src="https://weclouddata.s3.amazonaws.com/images/logos/wcd_logo_new_2.png" width='20%'>
</left>

<h1 align="left"> Exercise: Complete Multi-Agent Upgrade </h1>
<center align="left"> <font size='4'>  Developed by: </font><font size='4' color='#33AAFBD'>WeCloudData</font></center>
<br>

**Purpose:**  
Finalize **Mini Project 4** by building a complete multi-agent system that includes:
- Multiple agents with well-defined roles
- Task planning and decomposition
- Memory integration (shared state)
- Coordination using LangGraph

**Project Scenario:** Smart Content Creation System

Build a multi-agent system that creates a high-quality blog post series about **"Top 10 AI Tools for Productivity in 2026"**.

**Required Components:**
- Planner Agent
- Researcher Agent (with FAISS retrieval from custom dataset)
- Writer Agent
- Reviewer Agent
- Shared memory / state management

---

## 1. Setup

In [1]:
# Run this cell first
!pip install -q "langchain==0.3.*" "langchain-openai==0.2.*" "langchain-community==0.3.*" "langgraph==0.2.*" faiss-cpu duckduckgo-search --force-reinstall

import os
from getpass import getpass

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

print("Setup complete!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.2/94.2 kB 5.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.7/58.7 kB 5.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 41.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.7/153.7 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 54.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 62.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.6/

In [2]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated, List
import operator

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

print("LLM and tools ready")

/usr/local/lib/python3.12/dist-packages/langgraph/checkpoint/base/__init__.py:18: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


LLM and tools ready


## TODO: Create Custom Dataset + FAISS Vector Store

In [3]:
# TODO 1: Create your own sample dataset (about AI Productivity Tools 2026)
documents = [
    "Notion AI is the top productivity tool in 2026 with advanced AI writing and database automation.",
    "Cursor AI is leading in AI-powered code editing and developer productivity.",
    "Perplexity AI is the most popular AI search and research assistant.",
    "Gamma AI excels at creating beautiful AI-powered presentations.",
    "Mem AI offers intelligent note-taking with strong memory and recall features.",
    "Average productivity gain reported by users of top AI tools in 2026 is 38%.",
    "Key trend in 2026 is deep integration between AI tools and existing workflows."
]

embeddings = OpenAIEmbeddings()
vectorstore = FAISS.from_texts(documents, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

print("Vector store ready - complete the TODO above")

Vector store ready - complete the TODO above


## TODO: Define Shared State and Agents

In [4]:
class AgentState(TypedDict):
    messages: Annotated[List[str], operator.add]
    plan: str
    research_data: str
    draft: str
    final_report: str
    next: str

def planner_node(state):
    prompt = f"Create a clear 4-step plan for: {state['messages'][-1]}"
    plan = llm.invoke(prompt).content
    return {"plan": plan, "next": "researcher"}

def researcher_node(state):
    docs = retriever.invoke("Top AI productivity tools 2026")
    data = "\n".join([doc.page_content for doc in docs])
    return {"research_data": data, "next": "writer"}

def writer_node(state):
    draft = llm.invoke(f"Write a blog post using this plan and data:\n{state['plan']}\n\nData:\n{state['research_data']}").content
    return {"draft": draft, "next": "reviewer"}

def reviewer_node(state):
    final = llm.invoke(f"Review and improve this draft:\n{state['draft']}").content
    return {"final_report": final, "next": "END"}

print("State and nodes - complete the TODOs")

State and nodes - complete the TODOs


## TODO: Build the LangGraph Workflow

In [5]:
# TODO 3: Build the complete LangGraph workflow with proper edges and conditional routing

from langgraph.graph import StateGraph, END

# Build the complete LangGraph workflow
workflow = StateGraph(AgentState)

workflow.add_node("planner", planner_node)
workflow.add_node("researcher", researcher_node)
workflow.add_node("writer", writer_node)
workflow.add_node("reviewer", reviewer_node)

workflow.set_entry_point("planner")

workflow.add_edge("planner", "researcher")
workflow.add_edge("researcher", "writer")
workflow.add_edge("writer", "reviewer")
workflow.add_edge("reviewer", END)   # This is the correct way

graph = workflow.compile()

print("Multi-Agent system - complete the TODO")

Multi-Agent system - complete the TODO


## TODO: Run the Full System

In [6]:
def run_content_creation_system():
    query = "Create a high-quality blog post series on 'Top 10 AI Tools for Productivity in 2026'"
    result = graph.invoke({"messages": [query], "next": "planner"})

    print("\n" + "="*80)
    print("FINAL BLOG POST GENERATED")
    print("="*80)
    print(result.get("final_report", "No report generated"))

run_content_creation_system()


FINAL BLOG POST GENERATED
# Top 10 AI Tools for Productivity in 2026: A Comprehensive Blog Series

As we enter 2026, the productivity tools landscape has undergone a remarkable transformation, driven by significant advancements in artificial intelligence (AI). Users of leading AI tools report an average productivity gain of 38%, underscoring that these technologies are not merely a trend—they are essential for professionals, entrepreneurs, and students alike. In this blog series, we will explore the top 10 AI tools that are revolutionizing productivity this year.

## Step 1: Research and Content Planning

### Identify Target Audience
Before delving into the specifics of each tool, it’s essential to define our target audience. Our readers include professionals seeking to streamline their workflows, entrepreneurs looking for innovative solutions to enhance their business operations, and students aiming to maximize their study efficiency. Each group faces unique productivity challenges, 